# Cable-Driven Wrist — Kinematic Visualization
### 2-DOF tendon-driven wrist mechanism of the tensegrity manipulator

---

## Background and References

This notebook visualizes the kinematics of the **cable-driven 2-DOF wrist joint**
used in the FAPS tendon-driven manipulator, as described in:

> **Nemoto, T., Walter, J., Bachmann, C., & Raatz, A.** (2022). Highly Dynamic 2-DOF
> Cable-Driven Robotic Wrist Based on a Novel Topology. *IEEE Robotics and Automation
> Letters*, 7(2), 5727–5734. DOI: 10.1109/LRA.2022.3158382.

> **Klein, M.** (2023). *Arbeitsraumanalyse, Simulation und Bewegungsplanung eines
> seilgetriebenen robotischen Manipulators.* Master's thesis, Friedrich-Alexander-
> Universität Erlangen-Nürnberg (FAPS), **§3.2.4** (cable tension distribution).

Additional references:

- **Verhoeven, R.** (2004). *Analysis of the Workspace of Tendon-based Stewart
  Platforms.* PhD thesis, Universität Duisburg-Essen. — Structure matrix definition,
  n+1 full-control theorem.
- **Pott, A., Bruckmann, T., & Mikelsons, L.** (2009). Closed-Form Force Distribution
  for Parallel Wire Robots. In *Computational Kinematics*, Springer, pp. 25–34. —
  Closed-form tension distribution (Eq. 9 in Nemoto).
- **Friesen, J. M., Glick, P., Fanton, M., et al.** (2018). The Second Generation
  Prototype of a Duct Climbing Tensegrity Robot, DuCTTv2. *IEEE ICRA*. — Compliant
  tensegrity-inspired joints with passive cable constraints.

---

## Principle of the Cable-Driven Wrist

The wrist uses a **cable-driven parallel mechanism** with two categories of cables:

- **3 Passive cables** — connect the base ring to the **rotation center** (origin),
  forming an inverted tetrahedron. They provide kinematic constraint equivalent to
  a spherical joint: restricting 3 translational DOF while leaving exactly 2
  rotational DOF (tilt about X and Y axes). The passive cables have no motors.
- **3 Active cables** — driven by motors, connecting the base ring to the movable
  **platform** below the rotation center. With $m = 3$ active cables for $n = 2$
  DOF, the system is redundantly actuated (redundancy $r = m - n = 1$), enabling
  tension optimization.

The passive cables are arranged at **120° intervals** on the base ring
at angles 0°, 120°, 240°. The active cables alternate at 60°, 180°, 300°.
Both sets share the same base ring at radius $R$, elevated $h_b$ above the
rotation origin. The active cables terminate on the platform at radius $r$,
located at distance $h$ below the origin.

```
     Base ring (z = h_b, radius R)
   P₁ ○──────────────○ A₁
       │╲            ╱│
       │  ╲        ╱  │        P = passive cable anchors (base)
       │    ╲    ╱    │        A = active cable anchors (base)
       │      ╳       │
       │    ╱    ╲    │        Passive cables converge at O
       │  ╱        ╲  │        Active cables reach to platform
   A₃ ○╱────── ○ ───╲○ P₂
              O ← rotation center (origin, z = 0)
             ╱│╲
           ╱  │  ╲
         ╱    │    ╲           Active cables continue to
        a₃    a₂    a₁        platform anchors (z = −h, radius r)
       ○──────○──────○
         Platform ring
```

### Structure Matrix & Tension Distribution

For each active cable $i$, the base anchor is $\mathbf{b}_i$ and the platform
anchor (in world frame) is $\mathbf{p}_i(\Psi, \Theta)$. The cable unit vector
and moment arm are:

$$\mathbf{u}_i = \frac{\mathbf{b}_i - \mathbf{p}_i}{\|\mathbf{b}_i - \mathbf{p}_i\|}, \quad
\boldsymbol{\tau}_i = \mathbf{p}_i \times \mathbf{u}_i$$

The **structure matrix** $\mathbf{A} \in \mathbb{R}^{2 \times 3}$ maps active cable
tensions $\mathbf{t} \in \mathbb{R}^3$ to the wrist torque $\boldsymbol{\tau} \in \mathbb{R}^2$:

$$\boldsymbol{\tau} = \mathbf{A}\,\mathbf{t}, \qquad
A_{k,i} = \hat{\mathbf{e}}_k \cdot (\mathbf{p}_i \times \mathbf{u}_i), \quad k \in \{x, y\}$$

Given desired torque $\boldsymbol{\tau}^*$, the **closed-form tension distribution**
(Pott et al. 2009) with reference tension $\mathbf{t}_\mathrm{ref}$ is:

$$\mathbf{t}^* = \mathbf{t}_\mathrm{ref}
 + \mathbf{A}^\dagger\bigl(\boldsymbol{\tau}^* - \mathbf{A}\,\mathbf{t}_\mathrm{ref}\bigr)$$

The workspace is **feasible** at $(\Psi, \Theta)$ if $\mathbf{t}^* > 0$ holds
for the zero-torque case ($\boldsymbol{\tau}^* = \mathbf{0}$), i.e., all cables
remain taut under the reference pre-tension.

---

## Rotation Model

The platform orientation is parametrized by X–Y fixed Euler angles $(\Psi, \Theta)$:

$$R(\Psi, \Theta) = R_y(\Theta) \cdot R_x(\Psi)$$

Each platform anchor in the body frame $\mathbf{p}_i^B = (r\cos\alpha_i,\; r\sin\alpha_i,\; -h)$
transforms to the world frame via:

$$\mathbf{p}_i(\Psi, \Theta) = R(\Psi, \Theta) \cdot \mathbf{p}_i^B$$

The rotation center is at the origin — the convergence point of the 3 passive cables,
which sits $h_b$ below the base ring.

---

## Quick Start

> **Kernel:** requires `numpy`, `plotly`.
> Use the `env_isaaclab` conda environment:
> ```bash
> conda run -n env_isaaclab jupyter notebook
> ```
> Or set the VS Code kernel to `env_isaaclab (Python)`.

1. Run **Cell 2 (Parameters)** — adjust `R`, `r`, `h` as needed.
2. Run all remaining cells in order.
3. Click **▶ Play** in the animation output.

In [5]:

# =================================================================
# PARAMETERS — adjust here
# =================================================================
import sys
sys.path.insert(0, ".")

import numpy as np
from helpers.wrist import compute_anchors
from helpers.colors import (
    FAPS_GREEN, FAPS_BLUE, FAPS_GRAY, FAPS_LIGHT,
    ACTIVE_COL, PASSIVE_COL,
)

# ═══ DIMENSION SET ═══
# Choose: "nemoto" for Nemoto et al. (2022) prototype
#         "implemented" for the actual wrist on the FAPS manipulator
DIMENSION_SET = "implemented"

if DIMENSION_SET == "nemoto":
    R = 80.0       # Base anchor circle radius [mm]
    r = 20.0       # Platform anchor circle radius [mm]
    h = 75.0       # Vertical distance: rotation origin → platform (downward) [mm]
    h_b = 10.0     # Vertical distance: rotation origin → base ring (downward) [mm]
    tilt_deg = 55  # Approximate maximum horizontal reach of the wrist [deg]
elif DIMENSION_SET == "implemented":
    R = 72.5       # Same base radius
    r = 25.0       # As-implemented from tendon_actuator.py [mm]
    h = 76.0       # Same vertical distance
    h_b = 9.0     # Same base ring offset below rotation origin
    tilt_deg = 45  # As-implemented from tendon_actuator.py [deg]
else:
    raise ValueError(f"Unknown DIMENSION_SET: {DIMENSION_SET!r}")

# Workspace sweep range [deg]
tilt_max_deg = tilt_deg # 65.0

# Reference pre-tension for workspace analysis [N]
t_ref_val = 1.0

# Number of animation frames
n_frames = 120

# Amplitude of the animation sweep [deg]
sweep_amplitude_deg = tilt_deg # 40.0

# ═══ OPTIONS ═══
show_caption = True    # Set False for thesis-ready figures without titles

# =================================================================
# Derived constants — do not modify
# =================================================================
tilt_max_rad = np.radians(tilt_max_deg)
anchors = compute_anchors(R, r, h, h_b)
base_passive = anchors["base_passive"]
base_active = anchors["base_active"]
passive_convergence = anchors["passive_convergence"]
platform_active_body = anchors["platform_active_body"]

print(f"=== Wrist Dimensions ({DIMENSION_SET}) ===")
print(f"R   = {R:.1f} mm  (base ring radius)")
print(f"r   = {r:.1f} mm  (platform ring radius)")
print(f"h   = {h:.1f} mm  (rotation origin → platform, −z direction)")
print(f"h_b = {h_b:.1f} mm  (rotation origin → base ring, −z direction)")
print(f"Tilt range: ±{tilt_max_deg:.0f}°")
print()
print("Rotation origin (passive cables converge here): z = 0")
print(f"Base ring z = −h_b = {-h_b:.1f} mm")
print(f"Platform z  = −h   = {-h:.1f} mm")
print()

passive_angles = np.degrees(anchors["passive_base_angles"])
active_angles = np.degrees(anchors["active_base_angles"])
print("--- Passive cable anchors (blue) — base ring, converge at origin ---")
for i, (a, p) in enumerate(zip(passive_angles, base_passive)):
    print(f"  P{i+1}: ({p[0]:+7.2f}, {p[1]:+7.2f}, {p[2]:+.1f}) mm  @ {a:5.0f}°")

print("--- Active cable anchors (red) — base ring to platform ---")
for i, (a, p) in enumerate(zip(active_angles, base_active)):
    print(f"  A{i+1}: ({p[0]:+7.2f}, {p[1]:+7.2f}, {p[2]:+.1f}) mm  @ {a:5.0f}°")

print(f"\nPassive convergence point (rotation origin): {passive_convergence}")
print("\n--- Platform anchors, body frame (active) ---")
for i, p in enumerate(platform_active_body):
    print(f"  a{i+1}: ({p[0]:+7.2f}, {p[1]:+7.2f}, {p[2]:.1f}) mm")


=== Wrist Dimensions (implemented) ===
R   = 72.5 mm  (base ring radius)
r   = 25.0 mm  (platform ring radius)
h   = 76.0 mm  (rotation origin → platform, −z direction)
h_b = 9.0 mm  (rotation origin → base ring, −z direction)
Tilt range: ±45°

Rotation origin (passive cables converge here): z = 0
Base ring z = −h_b = -9.0 mm
Platform z  = −h   = -76.0 mm

--- Passive cable anchors (blue) — base ring, converge at origin ---
  P1: ( +72.50,   +0.00, -9.0) mm  @     0°
  P2: ( -36.25,  +62.79, -9.0) mm  @   120°
  P3: ( -36.25,  -62.79, -9.0) mm  @   240°
--- Active cable anchors (red) — base ring to platform ---
  A1: ( +36.25,  +62.79, -9.0) mm  @    60°
  A2: ( -72.50,   +0.00, -9.0) mm  @   180°
  A3: ( +36.25,  -62.79, -9.0) mm  @   300°

Passive convergence point (rotation origin): [0. 0. 0.]

--- Platform anchors, body frame (active) ---
  a1: ( +12.50,  +21.65, -76.0) mm
  a2: ( -25.00,   +0.00, -76.0) mm
  a3: ( +12.50,  -21.65, -76.0) mm


In [6]:

# =================================================================
# Kinematics Verification
# =================================================================
from helpers.wrist import (
    structure_matrix, tension_distribution, is_feasible,
    platform_anchors_world, cable_geometry,
)

# --- Zero-config verification ---
A_zero = structure_matrix(0, 0, base_active, platform_active_body)
rank_zero = np.linalg.matrix_rank(A_zero, tol=1e-8)
feasible_zero, t_zero = is_feasible(0, 0, base_active, platform_active_body)

_, _, Vt = np.linalg.svd(A_zero)
null_vec = Vt[-1]

passive_lengths = np.linalg.norm(base_passive - passive_convergence, axis=1)
_, L_active_zero, _ = cable_geometry(
    base_active, platform_anchors_world(0, 0, platform_active_body))

print("=== Verification at Zero Config ===")
print(f"Structure matrix A ({DIMENSION_SET}, R={R}, r={r}, h={h} mm):")
print(f"  [{A_zero[0, 0]:+10.4f}  {A_zero[0, 1]:+10.4f}  {A_zero[0, 2]:+10.4f}]  (τ_x)")
print(f"  [{A_zero[1, 0]:+10.4f}  {A_zero[1, 1]:+10.4f}  {A_zero[1, 2]:+10.4f}]  (τ_y)")
print(f"\nRank: {rank_zero}  (must be 2 for full 2-DOF control)")
print(f"Feasible at zero torque: {feasible_zero}")
print(f"Tensions: [{t_zero[0]:.4f}, {t_zero[1]:.4f}, {t_zero[2]:.4f}]")
print(f"Null-space basis: [{null_vec[0]:+.4f}, {null_vec[1]:+.4f}, {null_vec[2]:+.4f}]")
print(f"\n--- Cable lengths at zero config ---")
print(f"Passive (base→origin): {passive_lengths.round(2)} mm  (fixed, all equal)")
print(f"Active  (base→platform): {L_active_zero.round(2)} mm  (all equal by symmetry)")

# --- Workspace feasibility scan ---
n_scan = 200
psi_range = np.linspace(-tilt_max_rad, tilt_max_rad, n_scan)
theta_range = np.linspace(-tilt_max_rad, tilt_max_rad, n_scan)
feasible_map = np.zeros((n_scan, n_scan), dtype=bool)

for i, psi in enumerate(psi_range):
    for j, theta in enumerate(theta_range):
        ok, _ = is_feasible(psi, theta, base_active, platform_active_body)
        feasible_map[i, j] = ok

feasible_pct = 100 * feasible_map.sum() / feasible_map.size
print(f"\n=== Workspace Scan (±{tilt_max_deg:.0f}°) ===")
print(f"Feasible area: {feasible_pct:.1f}% of ±{tilt_max_deg:.0f}° square")

# --- Compare with as-implemented Jacobian transpose ---
jac_impl = np.array([
    [-0.013856,  0.0,     +0.013856],
    [+0.008,    -0.016,   +0.008],
])
from helpers.wrist import compute_anchors as _ca
impl_anchors = _ca(R, 16.0, h, h_b)
A_impl = structure_matrix(0, 0, impl_anchors["base_active"],
                          impl_anchors["platform_active_body"])
print(f"\n--- As-Implemented (r=16 mm) Structure Matrix ---")
print(f"  [{A_impl[0, 0]:+10.6f}  {A_impl[0, 1]:+10.6f}  {A_impl[0, 2]:+10.6f}]")
print(f"  [{A_impl[1, 0]:+10.6f}  {A_impl[1, 1]:+10.6f}  {A_impl[1, 2]:+10.6f}]")
print(f"\nJ^T from tendon_actuator.py:")
print(f"  [{jac_impl[0, 0]:+10.6f}  {jac_impl[0, 1]:+10.6f}  {jac_impl[0, 2]:+10.6f}]")
print(f"  [{jac_impl[1, 0]:+10.6f}  {jac_impl[1, 1]:+10.6f}  {jac_impl[1, 2]:+10.6f}]")


=== Verification at Zero Config ===
Structure matrix A (implemented, R=72.5, r=25.0, h=76.0 mm):
  [  +55.7284     +0.0000    -55.7284]  (τ_x)
  [  -32.1748    +64.3496    -32.1748]  (τ_y)

Rank: 2  (must be 2 for full 2-DOF control)
Feasible at zero torque: True
Tensions: [1.0000, 1.0000, 1.0000]
Null-space basis: [+0.5774, +0.5774, +0.5774]

--- Cable lengths at zero config ---
Passive (base→origin): [73.06 73.06 73.06] mm  (fixed, all equal)
Active  (base→platform): [82.13 82.13 82.13] mm  (all equal by symmetry)

=== Workspace Scan (±45°) ===
Feasible area: 100.0% of ±45° square

--- As-Implemented (r=16 mm) Structure Matrix ---
  [+53.023129   +0.000000  -53.023129]
  [-30.612918  +61.225835  -30.612918]

J^T from tendon_actuator.py:
  [ -0.013856   +0.000000   +0.013856]
  [ +0.008000   -0.016000   +0.008000]


In [7]:

# =================================================================
# Static 3D Overview
# =================================================================
from helpers.wrist_plots import wrist_static_figure

fig = wrist_static_figure(R, r, h, h_b, show_caption=show_caption)
fig.show()


In [8]:

# =================================================================
# Animated 3D Visualization
# =================================================================
from helpers.wrist_plots import wrist_animated_figure

fig = wrist_animated_figure(
    R, r, h, h_b,
    tilt_max_deg=tilt_max_deg,
    t_ref_val=t_ref_val,
    n_frames=n_frames,
    sweep_amplitude_deg=sweep_amplitude_deg,
    show_caption=show_caption,
)
fig.show()


---

## Summary

| Property | Nemoto (2022) Prototype | As-Implemented (Isaac Sim) |
|---|---|---|
| Base anchor radius $R$ | **80 mm** | — (constant J^T model) |
| Platform anchor radius $r$ | **20 mm** | **16 mm** |
| Vertical distance $h$ | **75 mm** | **~75 mm** |
| Passive cables | 3 (kinematic constraint) | None (2 revolute joints) |
| Active cables | 3 (120° spacing, 60° offset) | 3 (120° spacing at 60°, 180°, 300°) |
| DOF | 2 (Ψ about X, Θ about Y) | 2 (wrist_x, wrist_y revolute) |
| Joint limits | ±65° (workspace boundary) | ±0.8 rad ≈ ±45.8° |
| Redundancy | 1 (3 cables − 2 DOF) | 1 (same) |
| Torque model | Configuration-dependent $\mathbf{A}(\Psi, \Theta)$ | Constant $J^T$ (zero-config) |

### Cable Topology (Nemoto 2022)

The key innovation is the **hybrid passive + active cable topology**:

- **3 passive cables** at 0°, 120°, 240° connect base radius $R$ to platform
  radius $r$. These provide **kinematic constraint** equivalent to a spherical
  joint (3 translational DOF removed), without requiring a physical universal
  joint or gimbal. The passive cables have **no motors** — they are tensioned
  purely by the geometry.

- **3 active cables** at 60°, 180°, 300° are motor-driven and provide **full
  2-DOF control** of the platform orientation. With $m = 3 > n = 2$, the system
  has **one degree of redundancy**, allowing tension optimization via the
  null-space of the structure matrix.

### Tension Distribution

At any orientation $(\Psi, \Theta)$, the structure matrix
$\mathbf{A} \in \mathbb{R}^{2 \times 3}$ maps cable tensions to joint torques.
The closed-form solution (Pott et al. 2009):

$$\mathbf{t}^* = \mathbf{t}_\mathrm{ref} + \mathbf{A}^\dagger(\boldsymbol{\tau}^* - \mathbf{A}\,\mathbf{t}_\mathrm{ref})$$

ensures all tensions stay positive (feasible workspace) while minimizing
deviation from the reference pre-tension. The null-space component
$\mathbf{v} \in \ker(\mathbf{A})$ can shift all tensions uniformly without
affecting the net torque.

### Isaac Sim Implementation Note

The as-implemented model replaces the 3 passive cables + spherical equivalence
with **two sequential revolute joints** (`wrist_x_joint` about X, then
`wrist_y_joint` about Y), both at $z = -0.836$ m. The 3 active cables use a
**constant Jacobian-transpose mapping** (valid for the ±45° operating range):

$$J^T = \begin{bmatrix}
-0.013856 & 0 & +0.013856 \\
+0.008 & -0.016 & +0.008
\end{bmatrix}$$

Row 1 (wrist_y) produces torque about X; Row 2 (wrist_x) produces torque
about Y — the naming reflects the axis of *motion*, not the axis of *rotation*.

---

## References

1. **Nemoto, T., Walter, J., Bachmann, C., & Raatz, A.** (2022). Highly Dynamic
   2-DOF Cable-Driven Robotic Wrist Based on a Novel Topology. *IEEE Robotics and
   Automation Letters*, 7(2), 5727–5734. DOI: 10.1109/LRA.2022.3158382.

2. **Klein, M.** (2023). *Arbeitsraumanalyse, Simulation und Bewegungsplanung
   eines seilgetriebenen robotischen Manipulators.* Master's thesis, FAU
   Erlangen-Nürnberg (FAPS). §3.2.4 (cable tension distribution), §3.2
   (joint parameters).

3. **Verhoeven, R.** (2004). *Analysis of the Workspace of Tendon-based Stewart
   Platforms.* PhD thesis, Universität Duisburg-Essen. (Structure matrix,
   n+1 full-control theorem.)

4. **Pott, A., Bruckmann, T., & Mikelsons, L.** (2009). Closed-Form Force
   Distribution for Parallel Wire Robots. In *Computational Kinematics*,
   Springer, pp. 25–34. (Equation 9 in Nemoto.)

5. **Friesen, J. M., Glick, P., Fanton, M., et al.** (2018). The Second
   Generation Prototype of a Duct Climbing Tensegrity Robot, DuCTTv2.
   *IEEE ICRA*. (Compliant tensegrity joints with passive cable constraints.)

6. **Walter, J., Rothenbücher, M., & Raatz, A.** (2023). Development of a
   Tensegrity-Inspired Joint for Impact Isolation. In *Cable-Driven Parallel
   Robots (CableCon)*, Springer. (Tensegrity-inspired joint design.)